ACll server app4 index6 use korsi eitate


In [ ]:
!pip install open_clip_torch peft flask-cors pyngrok
!pip uninstall torchao -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/local_models
project_path = "/content/drive/MyDrive/MrNet-v1/MRNet-v1.0/"
for p in ['sagittal', 'coronal', 'axial']:
    !cp -r "{project_path}lora_{p}_best" /content/local_models/
    !cp "{project_path}attn_{p}_best.pth" /content/local_models/

Mounted at /content/drive


In [ ]:
%%writefile app.py
import os, io, base64, threading, traceback
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.amp import autocast
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
import open_clip
from peft import PeftModel
import matplotlib
matplotlib.use('Agg')

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

# ── Tell ngrok to skip its browser-warning interstitial on every response ─────
@app.after_request
def add_ngrok_header(response):
    response.headers['ngrok-skip-browser-warning'] = 'true'
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = '*'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Models are copied here from Drive in the Colab setup cell
MODEL_BASE_PATH   = os.environ.get("MODEL_BASE_PATH", "/content/local_models")
OPTIMAL_THRESHOLD = float(os.environ.get("OPTIMAL_THRESHOLD", "0.5"))
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[ACL-AI] Device : {DEVICE}")
print(f"[ACL-AI] Models : {MODEL_BASE_PATH}")
print(f"[ACL-AI] Threshold: {OPTIMAL_THRESHOLD}")

# ── MODEL DEFINITIONS (identical to your notebook Cell 1) ────────────────────
class GatedAttention(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256):
        super().__init__()
        self.attention_V       = nn.Linear(embed_dim, hidden_dim)
        self.attention_U       = nn.Linear(embed_dim, hidden_dim)
        self.attention_weights = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        a_v = torch.tanh(self.attention_V(x))
        a_u = torch.sigmoid(self.attention_U(x))
        w   = self.attention_weights(a_v * a_u)
        A   = torch.softmax(w, dim=0)
        M   = torch.sum(A * x, dim=0, keepdim=True)
        return M, A


class LoRABioMedCLIP(nn.Module):
    def __init__(self, lora_vision_encoder, base_clip_model, text_features):
        super().__init__()
        self.vision_encoder = lora_vision_encoder
        self.text_features  = text_features.detach()
        self.logit_scale    = base_clip_model.logit_scale
        self.attention      = GatedAttention(embed_dim=512, hidden_dim=256)

    def forward(self, image_volume):
        image_features = self.vision_encoder(image_volume)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        patient_image_feature, attn_weights = self.attention(image_features)
        patient_image_feature = patient_image_feature / patient_image_feature.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * patient_image_feature @ self.text_features.t()
        return logits, attn_weights


# ── GLOBAL MODEL CACHE ────────────────────────────────────────────────────────
clip_model    = None
preprocess    = None
tokenizer     = None
_model_loaded = False
_load_lock    = threading.Lock()


def load_base_model():
    """Load BioMedCLIP exactly once; thread-safe."""
    global clip_model, preprocess, tokenizer, _model_loaded
    with _load_lock:
        if _model_loaded:
            return
        print("[ACL-AI] Loading BioMedCLIP base model (~60 s on first call)…")
        model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        clip_model, _, preprocess = open_clip.create_model_and_transforms(model_name)
        clip_model   = clip_model.to(DEVICE)
        tokenizer    = open_clip.get_tokenizer(model_name)
        _model_loaded = True
        print("[ACL-AI] Base model loaded ✓")


# ── HELPERS ───────────────────────────────────────────────────────────────────
def get_ensembled_text_features(plane_name):
    """Identical to notebook Cell 1 — get_ensembled_text_features()."""
    prompts_healthy = [
        f"A normal {plane_name} MRI slice of the knee with an intact anterior cruciate ligament.",
        f"Healthy knee MRI showing a continuous ACL."
    ]
    prompts_tear = [
        f"A {plane_name} MRI slice of the knee showing a complete tear of the anterior cruciate ligament.",
        f"Ruptured anterior cruciate ligament with discontinuity."
    ]
    clip_model.eval()
    with torch.no_grad():
        tok_healthy  = tokenizer(prompts_healthy).to(DEVICE)
        feat_healthy = clip_model.encode_text(tok_healthy).mean(dim=0, keepdim=True)
        tok_tear     = tokenizer(prompts_tear).to(DEVICE)
        feat_tear    = clip_model.encode_text(tok_tear).mean(dim=0, keepdim=True)
        text_features = torch.cat([feat_healthy, feat_tear], dim=0)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


def load_patient_volume_from_bytes(npy_bytes):
    """
    Equivalent to notebook Cell 2 — load_patient_volume(),
    but accepts raw bytes instead of a file path (Colab in-memory upload).
    """
    vol = np.load(io.BytesIO(npy_bytes))
    vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)
    processed_slices = []
    for i in range(vol.shape[0]):
        slice_img = cv2.resize(vol[i], (224, 224))
        slice_rgb = np.stack(((slice_img * 255).astype(np.uint8),) * 3, axis=-1)
        processed_slices.append(preprocess(Image.fromarray(slice_rgb)))
    return torch.stack(processed_slices), vol


def generate_better_heatmap(single_slice, model, text_feats):
    """
    HIGH-QUALITY saliency — exact copy of notebook Cell 8.
    Steps: ReLU gradients → L2 norm → strong Gaussian smoothing →
           power-law transform → robust normalization → JET colormap.
    """
    model.vision_encoder.zero_grad()

    input_slice   = single_slice.clone().detach().requires_grad_(True)
    image_features = model.vision_encoder(input_slice)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    tear_feature   = text_feats[1:2]
    logit_scale    = model.logit_scale.exp()
    similarity     = (logit_scale * image_features @ tear_feature.t()).squeeze()
    similarity.backward()

    grad     = input_slice.grad.data.squeeze().cpu().numpy()
    saliency = np.sqrt(np.mean(np.square(np.maximum(grad, 0)), axis=0))   # L2 + ReLU
    saliency = cv2.GaussianBlur(saliency, (31, 31), 10)                   # strong smoothing
    saliency = np.power(saliency, 2)                                       # power-law

    v_min, v_max = saliency.min(), np.percentile(saliency, 99)            # robust norm
    if v_max > v_min:
        saliency = np.clip((saliency - v_min) / (v_max - v_min), 0, 1)
    else:
        saliency = np.zeros_like(saliency)

    return cv2.applyColorMap(np.uint8(255 * saliency), cv2.COLORMAP_JET)


def get_plane_analysis(npy_bytes, plane_name):
    """
    Exact logic of notebook Cell 8 — updated get_plane_analysis(),
    adapted to accept bytes instead of a patient_id + directory.

    Returns: (prob_tear, target_slice_idx, heatmap_bgr, vol_raw)
    """
    text_feats   = get_ensembled_text_features(plane_name)
    adapter_path = os.path.join(MODEL_BASE_PATH, f"lora_{plane_name}_best")
    attn_path    = os.path.join(MODEL_BASE_PATH, f"attn_{plane_name}_best.pth")

    # Clean any leftover PEFT state from the previous request
    if hasattr(clip_model.visual, "peft_config"):
        delattr(clip_model.visual, "peft_config")

    lora_vision = PeftModel.from_pretrained(clip_model.visual, adapter_path)
    model       = LoRABioMedCLIP(lora_vision, clip_model, text_feats).to(DEVICE)
    model.attention.load_state_dict(torch.load(attn_path, map_location=DEVICE))
    model.eval()

    img_tensor, vol_raw = load_patient_volume_from_bytes(npy_bytes)
    img_tensor_device   = img_tensor.to(DEVICE)

    with torch.no_grad():
        with autocast('cuda'):
            logits, attn_weights = model(img_tensor_device)
            prob_tear            = logits.softmax(dim=-1)[0, 1].item()
            target_slice_idx     = torch.argmax(attn_weights).item()

    # Generate high-quality heatmap OUTSIDE no_grad block (needs gradients)
    single_slice = img_tensor_device[target_slice_idx:target_slice_idx + 1]
    heatmap_color = generate_better_heatmap(single_slice, model, text_feats)

    # Unload LoRA adapter to free GPU memory between planes
    clip_model.visual = lora_vision.unload()

    return prob_tear, target_slice_idx, heatmap_color, vol_raw


def arr_to_b64png(arr_rgb):
    """HxWx3 uint8 ndarray → base64 PNG string (for embedding in JSON)."""
    buf = io.BytesIO()
    Image.fromarray(arr_rgb).save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()


def build_visual_panel(vol_raw, idx, heat_bgr):
    """
    Equivalent to notebook Cell 9 — the 3-column visual dashboard per plane.
    Returns dict with 'original', 'heatmap', 'overlay' as base64 PNGs.
    """
    slice_img_resized = cv2.resize(vol_raw[idx], (224, 224))
    slice_rgb         = np.stack(((slice_img_resized * 255).astype(np.uint8),) * 3, axis=-1)
    heat_rgb          = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    overlay           = cv2.addWeighted(slice_rgb, 0.5, heat_rgb, 0.5, 0)
    return {
        'original': arr_to_b64png(slice_rgb),
        'heatmap':  arr_to_b64png(heat_rgb),
        'overlay':  arr_to_b64png(overlay),
    }


def build_clinical_report_text(patient_id, ensemble_prob, threshold,
                                s_prob, s_idx, c_prob, c_idx, a_prob, a_idx):
    """
    Equivalent to notebook Cell 9 — generate_clinical_report() text block.
    Returns a plain-text / markdown string (sent as JSON field).
    """
    pred_status = "ACL Tear" if ensemble_prob >= threshold else "Intact/Healthy"
    risk_level  = ("High" if ensemble_prob * 100 >= 80
                   else "Moderate" if ensemble_prob >= threshold else "Low")
    return {
        "patient_id":    patient_id,
        "pred_status":   pred_status,
        "risk_level":    risk_level,
        "ensemble_pct":  round(ensemble_prob * 100, 1),
        "threshold_pct": round(threshold * 100, 0),
        "sagittal_pct":  round(s_prob * 100, 1),
        "sagittal_slice": s_idx,
        "coronal_pct":   round(c_prob * 100, 1),
        "coronal_slice": c_idx,
        "axial_pct":     round(a_prob * 100, 1),
        "axial_slice":   a_idx,
        "interpretation": (
            f"The multi-planar BioMedCLIP ensemble returned a confidence score of "
            f"{ensemble_prob*100:.1f}%, which "
            + ("exceeds" if ensemble_prob >= threshold else "falls below")
            + f" the decision threshold of {threshold*100:.0f}%. "
            + ("This indicates a statistically significant likelihood of Anterior Cruciate "
               "Ligament disruption. The sagittal plane (60% ensemble weight) is the primary "
               "contributor to this result. Clinical correlation and physician review are strongly "
               "recommended before any treatment decision."
               if ensemble_prob >= threshold else
               "This indicates a low probability of ACL injury. The ligament appears structurally "
               "intact across all three imaging planes. Routine clinical follow-up is advisable "
               "at the physician's discretion.")
        )
    }


# ── ROUTES ────────────────────────────────────────────────────────────────────
@app.route('/health', methods=['GET', 'OPTIONS'])
def health():
    """Liveness check — the UI Connect button calls this."""
    return jsonify({'status': 'ok', 'device': str(DEVICE), 'threshold': OPTIMAL_THRESHOLD})


@app.route('/analyze', methods=['POST', 'OPTIONS'])
def analyze():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    """
    POST multipart/form-data with:
        sagittal   — .npy file
        coronal    — .npy file
        axial      — .npy file
        threshold  — float (optional, defaults to OPTIMAL_THRESHOLD)
        patient_id — string (optional)

    Returns full JSON matching index.html UI contract.
    """
    load_base_model()

    # ── Validate files ────────────────────────────────────────────────────────
    for plane in ('sagittal', 'coronal', 'axial'):
        if plane not in request.files:
            return jsonify({'error': f'Missing file: {plane}'}), 400

    threshold  = float(request.form.get('threshold', OPTIMAL_THRESHOLD))
    patient_id = request.form.get('patient_id', 'Unknown')

    # Read bytes in memory — no /tmp needed on Colab
    sag_bytes = request.files['sagittal'].read()
    cor_bytes = request.files['coronal'].read()
    ax_bytes  = request.files['axial'].read()

    try:
        print(f"[ACL-AI] ── Analyzing patient: {patient_id} ──────────────")

        s_prob, s_idx, s_heat, s_vol = get_plane_analysis(sag_bytes, 'sagittal')
        print(f"[ACL-AI]  Sagittal : {s_prob*100:.1f}%  (top slice #{s_idx})")

        c_prob, c_idx, c_heat, c_vol = get_plane_analysis(cor_bytes, 'coronal')
        print(f"[ACL-AI]  Coronal  : {c_prob*100:.1f}%  (top slice #{c_idx})")

        a_prob, a_idx, a_heat, a_vol = get_plane_analysis(ax_bytes, 'axial')
        print(f"[ACL-AI]  Axial    : {a_prob*100:.1f}%  (top slice #{a_idx})")

    except FileNotFoundError as e:
        return jsonify({'error': f'Model file not found: {e}. Did the setup cell run?'}), 500
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

    # ── Weighted ensemble (0.6 / 0.3 / 0.1 — matches notebook Cell 9) ─────────
    ensemble  = (0.6 * s_prob) + (0.3 * c_prob) + (0.1 * a_prob)
    diagnosis = "ACL Tear Detected" if ensemble >= threshold else "Healthy / Intact ACL"
    print(f"[ACL-AI]  Ensemble : {ensemble*100:.1f}%  →  {diagnosis}")

    # ── Visual panels (original + heatmap + overlay per plane) ────────────────
    report = build_clinical_report_text(
        patient_id, ensemble, threshold,
        s_prob, s_idx, c_prob, c_idx, a_prob, a_idx
    )

    return jsonify({
        # ── Primary fields the UI reads ──────────────────────────────────────
        'diagnosis':  diagnosis,
        'confidence': round(ensemble, 6),    # raw 0–1 float; UI multiplies × 100

        # ── Full clinical report (notebook Cell 9 equivalent) ────────────────
        'report': report,

        # ── Extra metadata ───────────────────────────────────────────────────
        'patient_id': patient_id,
        'threshold':  threshold,

        # ── Per-plane detail: probability + top slice + 3 base64 images ──────
        'planes': {
            'sagittal': {'probability': round(s_prob * 100, 1), 'top_slice': s_idx,
                         **build_visual_panel(s_vol, s_idx, s_heat)},
            'coronal':  {'probability': round(c_prob * 100, 1), 'top_slice': c_idx,
                         **build_visual_panel(c_vol, c_idx, c_heat)},
            'axial':    {'probability': round(a_prob * 100, 1), 'top_slice': a_idx,
                         **build_visual_panel(a_vol, a_idx, a_heat)},
        }
    })


# ── ENTRY POINT ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # threaded=False is critical — PyTorch / CUDA is not thread-safe
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=False)


Writing app.py


In [ ]:
from pyngrok import conf
conf.get_default().auth_token = "3DTudsXOILvOgALXoUMOBniIDQx_5u2hGfzVuJt2522DCjWyA"

# This disables the interstitial at the ngrok level
from pyngrok import ngrok
ngrok.set_auth_token("3DTudsXOILvOgALXoUMOBniIDQx_5u2hGfzVuJt2522DCjWyA")
tunnel = ngrok.connect(5000, bind_tls=True)
print(tunnel.public_url)

# Then start Flask
!python app.py

https://populate-legible-shiny.ngrok-free.dev
[ACL-AI] Device : cpu
[ACL-AI] Models : /content/local_models
[ACL-AI] Threshold: 0.5
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [09/May/2026 14:21:23] "OPTIONS /health HTTP/1.1" 200 -
127.0.0.1 - - [09/May/2026 14:21:23] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [09/May/2026 14:23:01] "OPTIONS /analyze HTTP/1.1" 200 -
[ACL-AI] Loading BioMedCLIP base model (~60 s on first call)…
open_clip_config.json: 100% 707/707 [00:00<00:00, 3.34MB/s]
open_clip_pytorch_model.bin: 100% 784M/784M [00:06<00:00, 126MB/s]
config.json: 100% 385/385 [00:00<00:00, 2.36MB/s]
tokenizer_config.json: 100% 28.0/28.0 [00:00<00:00, 113kB/s]
vocab.txt: 225kB [00:00, 25.8MB/s]
[ACL-AI] Base model loaded ✓
[ACL-AI] ── Analyzing patient: 0175 ──────────────
/content/app.py:187: UserWarning: CUDA is not available or torch_xl